# Using both categorical and numerical data

In [2]:
import pandas as pd

adult_census = pd.read_csv("data/adult_data.csv")
# drop the duplicated column `"education-num"` as stated in the first notebook
adult_census = adult_census.drop(columns="education-num")

target_name = "class"
target = adult_census[target_name]

data = adult_census.drop(columns=[target_name])

In [3]:
from sklearn.compose import make_column_selector as selector

numerical_columns_selector = selector(dtype_exclude=object)
categorical_columns_selector = selector(dtype_include=object)

numerical_columns = numerical_columns_selector(data)
categorical_columns = categorical_columns_selector(data)

In [7]:
data[numerical_columns].head()

,age,fnlwgt,capital-gain,capital-loss,hours-per-week
0,25.0,226802.0,0.0,0.0,40.0
1,38.0,89814.0,0.0,0.0,50.0
2,28.0,336951.0,0.0,0.0,40.0
3,44.0,160323.0,7688.0,0.0,40.0
4,18.0,103497.0,0.0,0.0,30.0


In [9]:
data[categorical_columns].head()

,workclass,education,marital-status,occupation,relationship,race,sex,native-country
0,b'Private',b'11th',b'Never-married',b'Machine-op-inspct',b'Own-child',b'Black',b'Male',b'United-States'
1,b'Private',b'HS-grad',b'Married-civ-spouse',b'Farming-fishing',b'Husband',b'White',b'Male',b'United-States'
2,b'Local-gov',b'Assoc-acdm',b'Married-civ-spouse',b'Protective-serv',b'Husband',b'White',b'Male',b'United-States'
3,b'Private',b'Some-college',b'Married-civ-spouse',b'Machine-op-inspct',b'Husband',b'Black',b'Male',b'United-States'
4,b'?',b'Some-college',b'Never-married',b'?',b'Own-child',b'White',b'Female',b'United-States'


In [10]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_preprocessor = OneHotEncoder(handle_unknown="ignore")
numerical_preprocessor = StandardScaler()

- Create `ColumnTransfomer` by specifying three values: the preprocessor name, the transformer, and the columns.

In [12]:
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer(
    [
        ("one-hot-encoder", categorical_preprocessor, categorical_columns),
        ("standard_scaler", numerical_preprocessor, numerical_columns),
    ]
)

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

model = make_pipeline(preprocessor, LogisticRegression(max_iter = 500))
model

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('one-hot-encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass', 'education',
                                                   'marital-status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native-country']),
                                                 ('standard_scaler',
                                                  StandardScaler(),
                                                  ['age', 'fnlwgt',
                                                   'capital-gain',
                                                   'capital-loss',
                                                   'hours-per-week'])])),
                ('logisticregression', LogisticRegression(max_iter=500))])

> **Using train_test_split**

In [17]:
from sklearn.model_selection import train_test_split

data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state = 42)

In [18]:
_ = model.fit(data_train, target_train)

In [19]:
data_test.head()

,age,workclass,fnlwgt,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
7762,56.0,b'Private',33115.0,b'HS-grad',b'Divorced',b'Other-service',b'Unmarried',b'White',b'Female',0.0,0.0,40.0,b'United-States'
23881,25.0,b'Private',112847.0,b'HS-grad',b'Married-civ-spouse',b'Transport-moving',b'Own-child',b'Other',b'Male',0.0,0.0,40.0,b'United-States'
30507,43.0,b'Private',170525.0,b'Bachelors',b'Divorced',b'Prof-specialty',b'Not-in-family',b'White',b'Female',14344.0,0.0,40.0,b'United-States'
28911,32.0,b'Private',186788.0,b'HS-grad',b'Married-civ-spouse',b'Transport-moving',b'Husband',b'White',b'Male',0.0,0.0,40.0,b'United-States'
19484,39.0,b'Private',277886.0,b'Bachelors',b'Married-civ-spouse',b'Sales',b'Wife',b'White',b'Female',0.0,0.0,30.0,b'United-States'


In [23]:
model.predict(data_test)

array(["b'<=50K'", "b'<=50K'", "b'>50K'", ..., "b'<=50K'", "b'<=50K'",
       "b'>50K'"], dtype=object)

In [22]:
target_test[:5]

7762     b'<=50K'
23881    b'<=50K'
30507     b'>50K'
28911    b'<=50K'
19484    b'<=50K'
Name: class, dtype: object

In [24]:
model.score(data_test, target_test)

0.8586520350503645

> **Using cross validation**

In [25]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(model, data, target, cv = 5)
cv_results

{'fit_time': array([0.77252674, 0.58816361, 0.60066271, 0.61773562, 0.61085033]),
 'score_time': array([0.07044291, 0.06798244, 0.06558919, 0.0712657 , 0.07098246]),
 'test_score': array([0.85259494, 0.85095711, 0.84920147, 0.8531941 , 0.85657248])}

In [26]:
scores = cv_results["test_score"]
print(
    "The mean cross-validation accuracy is: "
    f"{scores.mean():.3f} ± {scores.std():.3f}"
)

The mean cross-validation accuracy is: 0.853 ± 0.002


- Using only numerical data acurracy was `0.800 ± 0.003`
- Using only categorical data with OneHotEncoding accuracy was `0.833 ± 0.003`

### Thus, using both data gives more accurate results.

---
 ###  Using `gradient boosting` methods to improve accuracy

In [28]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

categorical_preprocessor = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1
)

preprocessor = ColumnTransformer(
    [("categorical", categorical_preprocessor, categorical_columns)],
    remainder="passthrough",
)

model = make_pipeline(preprocessor, HistGradientBoostingClassifier())
model

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categorical',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['workclass', 'education',
                                                   'marital-status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native-country'])])),
                ('histgradientboostingclassifier',
                 HistGradientBoostingClassifier())])

- do not need to scale the numerical features
- using an ordinal encoding for the categorical variables is fine even if the encoding results in an arbitrary ordering

In [29]:
%%time
_ = model.fit(data_train, target_train)

CPU times: total: 4.64 s
Wall time: 3.44 s


In [30]:
model.score(data_test, target_test)

0.8815002866268119

- Thus, Gradient Boosted Machines are very popular for tabular data work.

### Which encoding to use :
 |                  | Meaningful order              | Non-meaningful order |
 | ---------------- | ----------------------------- | -------------------- |
 | Tree-based model | `OrdinalEncoder`              | `OrdinalEncoder`     |
 | Linear model     | `OrdinalEncoder` with caution | `OneHotEncoder`      |

 - `OneHotEncoder`: always does something meaningful, but can be unnecessary
   slow with trees.
 - `OrdinalEncoder`: can be detrimental for linear models unless your category
   has a meaningful order and you make sure that `OrdinalEncoder` respects this
   order. Trees can deal with `OrdinalEncoder` fine as long as they are deep
   enough.
 